## Воспроизводимое решение X5 Stage 2

Этот ноутбук последовательно создаёт финальный файл `test.csv` для задачи A: прогноз РТО магазинов на март 2025.

Рядом с ноутбуком должны лежать:

- `train_2.csv`
- `UPLOAD_9067_FINE_A014_A020_T0p75.csv`
- `UPLOAD_H2_RECENCY_WEIGHTED_REGION_TRANSFER_CANDIDATE.csv`
- `UPLOAD_9080_TRANSFER_BIAS_V1.csv`
- `UPLOAD_PLATEAU_MEDIAN_ENSEMBLE_V1.csv`

Итоговый прогноз строится как детерминированный лог-бленд нескольких промежуточных прогнозов поверх базового anchor. При запуске всех ячеек сверху вниз получается тот же `test.csv`, который был загружен в Контест.

### 1. Версия Python и список библиотек

Это обязательное условие воспроизводимости: выводим версию Python и список установленных пакетов.

In [1]:
!python -V

Python 3.11.9


In [1]:
!pip list

Package                   Version     Editable project location
------------------------- ----------- -------------------------------------------------------------------
absl-py                   2.4.0
altair                    6.1.0
annotated-doc             0.0.4
annotated-types           0.7.0
anyio                     4.13.0
asttokens                 3.0.1
astunparse                1.6.3
attrs                     26.1.0
blinker                   1.9.0
cachetools                7.0.6
catboost                  1.2.10
certifi                   2026.4.22
charset-normalizer        3.4.7
click                     8.3.3
colorama                  0.4.6
comm                      0.2.3
contourpy                 1.3.3
cycler                    0.12.1
debugpy                   1.8.20
decorator                 5.2.1
distro                    1.9.0
dm-tree                   0.1.10
docstring_parser          0.18.0
einops                    0.8.2
etils                     1.14.0
executing         

### 2. Импорты и фиксация random seed

Финальная сборка ниже не использует случайность, но seed фиксируется явно, чтобы решение соответствовало требованиям конкурса.

In [2]:
import os
import random
import hashlib
from pathlib import Path

import numpy as np
import pandas as pd

In [3]:
SEED = 2026
random.seed(SEED)
np.random.seed(SEED)
print('SEED =', SEED)

SEED = 2026


### 3. Загрузка данных и промежуточных прогнозов

`train_2.csv` используется для проверки набора магазинов и формата. Остальные CSV - это сопутствующие артефакты решения, из которых детерминированно собирается финальная посылка.

In [4]:
BASE = Path.cwd()

TRAIN_FILE = BASE / 'train_2.csv'
ANCHOR_FILE = BASE / 'UPLOAD_9067_FINE_A014_A020_T0p75.csv'
H2_FILE = BASE / 'UPLOAD_H2_RECENCY_WEIGHTED_REGION_TRANSFER_CANDIDATE.csv'
TRANSFER_FILE = BASE / 'UPLOAD_9080_TRANSFER_BIAS_V1.csv'
MEDIAN_FILE = BASE / 'UPLOAD_PLATEAU_MEDIAN_ENSEMBLE_V1.csv'

FINAL_FILE = BASE / 'test.csv'
EXPECTED_MD5 = '25232f8b8fd6171e3168d6d22e0c012d'

for path in [TRAIN_FILE, ANCHOR_FILE, H2_FILE, TRANSFER_FILE, MEDIAN_FILE]:
    if not path.exists():
        raise FileNotFoundError(f'Missing file: {path.name}')

train = pd.read_csv(TRAIN_FILE)
print('train shape:', train.shape)
print('unique shops:', train['new_id'].nunique())

train shape: (485082, 24)
unique shops: 18657


In [5]:
def read_submission(path, anchor=None):
    df = pd.read_csv(path).sort_values('new_id').reset_index(drop=True)
    if list(df.columns) != ['new_id', 'rto']:
        raise ValueError(f'{path.name}: expected columns new_id,rto')
    if df.isna().any().any():
        raise ValueError(f'{path.name}: contains NaN')
    if not (df['rto'] > 0).all():
        raise ValueError(f'{path.name}: contains non-positive rto')
    if df['new_id'].duplicated().any():
        raise ValueError(f'{path.name}: contains duplicated new_id')
    if anchor is not None and not df['new_id'].astype(int).equals(anchor['new_id'].astype(int)):
        raise ValueError(f'{path.name}: new_id set/order differs from anchor')
    return df

anchor = read_submission(ANCHOR_FILE)
h2 = read_submission(H2_FILE, anchor)
transfer = read_submission(TRANSFER_FILE, anchor)
median = read_submission(MEDIAN_FILE, anchor)

train_ids = train['new_id'].drop_duplicates().sort_values().reset_index(drop=True).astype(int)
assert anchor['new_id'].astype(int).equals(train_ids)
print('All input forecasts are valid and have the same store list.')

All input forecasts are valid and have the same store list.


### 4. Финальная формула прогноза

Финальный прогноз считается в лог-пространстве:

`final = anchor * exp(0.15 * log(h2 / anchor) + 1.00 * log(transfer / anchor) + 1.50 * log(median / anchor))`

Такой способ сохраняет положительность прогноза и применяет небольшие мультипликативные корректировки к anchor.

In [6]:
a = anchor['rto'].to_numpy(float)

log_h2 = np.log(h2['rto'].to_numpy(float) / a)
log_transfer = np.log(transfer['rto'].to_numpy(float) / a)
log_median = np.log(median['rto'].to_numpy(float) / a)

pred = a * np.exp(0.15 * log_h2 + 1.00 * log_transfer + 1.50 * log_median)
pred = np.maximum(pred, 1.0)

submission = pd.DataFrame({
    'new_id': anchor['new_id'].astype(int),
    'rto': pred,
})

submission.to_csv(FINAL_FILE, index=False, encoding='utf-8', lineterminator='\n', float_format='%.2f')
print('Saved:', FINAL_FILE.name)

Saved: test.csv


### 5. Проверка формата и точного совпадения

Проверяем все требования к `test.csv`: 18 657 строк, колонки `new_id,rto`, нет пропусков, прогнозы положительные, размер меньше 1 MB, а `new_id` совпадают с `train_2.csv`. Также проверяем md5 финального файла.

In [7]:
def file_md5(path):
    return hashlib.md5(Path(path).read_bytes()).hexdigest()

check = pd.read_csv(FINAL_FILE).sort_values('new_id').reset_index(drop=True)

assert len(check) == 18657
assert list(check.columns) == ['new_id', 'rto']
assert check['new_id'].astype(int).equals(train_ids)
assert not check.isna().any().any()
assert (check['rto'] > 0).all()
assert check['new_id'].duplicated().sum() == 0
assert os.path.getsize(FINAL_FILE) < 1_000_000

current_md5 = file_md5(FINAL_FILE)
print('rows:', len(check))
print('columns:', list(check.columns))
print('file size:', os.path.getsize(FINAL_FILE))
print('md5:', current_md5)

assert current_md5 == EXPECTED_MD5, 'Final test.csv does not match the uploaded Contest file'
print('OK: test.csv was reproduced exactly.')

rows: 18657
columns: ['new_id', 'rto']
file size: 333694
md5: 25232f8b8fd6171e3168d6d22e0c012d
OK: test.csv was reproduced exactly.


### 6. Статистика отличий от anchor

Эта ячейка не нужна для формирования файла, но она показывает, что финальная корректировка мала и контролируема.

In [8]:
ratio = check['rto'].to_numpy(float) / anchor['rto'].to_numpy(float)
abs_rel = np.abs(ratio - 1.0)

print('changed rows:', int((abs_rel > 1e-12).sum()))
print('changed share, %:', (abs_rel > 1e-12).mean() * 100)
print('mean abs rel change, %:', abs_rel.mean() * 100)
print('p95 abs rel change, %:', np.quantile(abs_rel, 0.95) * 100)
print('p99 abs rel change, %:', np.quantile(abs_rel, 0.99) * 100)
print('max abs rel change, %:', abs_rel.max() * 100)

changed rows: 3407
changed share, %: 18.261242429115075
mean abs rel change, %: 0.014692815375782401
p95 abs rel change, %: 0.08892421079066411
p99 abs rel change, %: 0.2243641147229023
max abs rel change, %: 0.7335693108649677
